# IMPORTS

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

%load_ext autoreload
import os, sys
import numpy as np

%matplotlib notebook
import matplotlib as mpl
import matplotlib.pyplot as plt
plt.style.use('default')
plt.close('all')

In [ ]:
import scipy as sp

def sample_latin_hypercube(dict_bounds, N_points=3000, seed=0):
    logging.info('Generating Latin Hypercube samples...')
    sample = sp.stats.qmc.LatinHypercube(d=len(dict_bounds), seed=seed).random(n=N_points)
    
    l_bounds = []
    u_bounds = []
    for key in dict_bounds.keys():
        l_bounds.append(dict_bounds[key][0])
        u_bounds.append(dict_bounds[key][1])
    
    theta_latin_hypercube = sp.stats.qmc.scale(sample, l_bounds, u_bounds)
    
    logging.debug('Sampled points: %s', theta_latin_hypercube.shape)
    return theta_latin_hypercube
    

def generate_toymodel_dataset(
    NN_samples_cosmo,
    NN_samples_augs,
    NN_noise_realizations,
    dict_bounds_cosmo=None,
    dict_bounds_augs=None,
    seed=0,
    path_save=None,
    model_name="ModelA",
    kcut=0.6,
    ks=3,
    kmin=0,
    kmax=1,
    NN_k=140
):

    logging.info('Generating toymodel dataset...')
    
    if dict_bounds_cosmo is None:
        dict_bounds_cosmo = {
            'A': [-0.9, -0.1],
            'B': [1.6, 2.0]
        }
        
    if dict_bounds_augs is None:
        dict_bounds_augs = {
            'C': [-2., 2.]
        }

    # Sample cosmological parameter space
    dict_bounds_sweep = {}
    for key in dict_bounds_cosmo.keys():
        if isinstance(dict_bounds_cosmo[key], list):
            assert len(dict_bounds_cosmo[key]) == 2, "Please provide bounds in the format 'param_name = [min, max]'"
            dict_bounds_sweep[key] = dict_bounds_cosmo[key]

    if dict_bounds_sweep:
        cosmos = sample_latin_hypercube(dict_bounds_sweep, N_points=NN_samples_cosmo, seed=seed)
    else:
        cosmos = np.zeros((NN_samples_cosmo, len(dict_bounds_cosmo.keys())))
        for ii, key in enumerate(dict_bounds_cosmo.keys()):
            cosmos[:, ii] = np.repeat(dict_bounds_cosmo[key], NN_samples_cosmo)
    
    # Sample augmentation parameter space
    dict_bounds_sweep = {}
    for key in dict_bounds_augs.keys():
        if isinstance(dict_bounds_augs[key], list):
            assert len(dict_bounds_augs[key]) == 2, "Please provide bounds in the format 'param_name = [min, max]'"
            dict_bounds_sweep[key] = dict_bounds_augs[key]
    
    if dict_bounds_sweep:
        aug_params = sample_latin_hypercube(dict_bounds_sweep, N_points=NN_samples_cosmo * NN_samples_augs, seed=seed)
        np.random.shuffle(aug_params)
    else:
        aug_params = np.zeros((NN_samples_cosmo * NN_samples_augs, len(dict_bounds_sweep.keys())))
        for ii, key in enumerate(dict_bounds_sweep.keys()):
            aug_params[:, ii] = dict_bounds_sweep[key]

    aug_params = np.reshape(aug_params, (NN_samples_cosmo, NN_samples_augs, aug_params.shape[-1]))

    # Generate observations with baccoemu
    
    xx = toy_model(cosmos[:, 0], cosmos[:, 1], aug_params[..., 0], kcut=kcut, ks=ks, kmin=kmin, kmax=kmax, NN_k=NN_k)

    scale_errors = np.linspace(0.02, 0.001, NN_k)
    xx = np.random.normal(loc=xx, scale=scale_errors, size=(NN_noise_realizations,)+xx.shape)
    
    # # Save data
    # if path_save is not None:
    #     if not os.path.exists(path_save):
    #         os.makedirs(path_save)
    #     np.save(os.path.join(path_save, model_name + '_cosmos.npy'), cosmos)
    #     np.save(os.path.join(path_save, model_name + '_xx.npy'), xx)
    #     np.save(os.path.join(path_save, model_name + '_aug_params.npy'), aug_params)
    
    return cosmos, xx, aug_params


def toy_model(AA, BB, CC, kcut=0.6, ks=3, kmin=0, kmax=1, NN_k=140):
    
    NN_cosmo = len(AA)
    NN_augs = CC.shape[1]

    AA = np.tile(AA[:, np.newaxis, np.newaxis], [1, NN_augs, NN_k])
    BB = np.tile(BB[:, np.newaxis, np.newaxis], [1, NN_augs, NN_k])
    CC = np.tile(CC[..., np.newaxis], [1, 1, NN_k])

    kk = np.linspace(kmin, kmax, NN_k)
    kk = np.tile(kk[np.newaxis, np.newaxis], [NN_cosmo, NN_augs, 1])

    ff = 0.5 * (
        np.tanh(10**ks * (kk - kcut)) * ((kk * (CC - AA)) - kcut * (CC - AA)) 
        + kk * (CC + AA) - kcut * (CC - AA) + 2 * BB
    )
    
    return ff

# SETUP TO GENERATE DATASETS

##### Define in "path_save_root" the path where the datasets will be stored. Also define boxsize and kmax

In [ ]:
kcut=0.6
ks=3.
kmin=0
kmax=1
NN_k=60
path_save_root = os.path.join("/cosmos_storage/home/dlopez/Projects/CL_inference/DATASETS_toymodel")

##### Define dictionaries with cosmological prior ranges for training, validation and test sets. Also number of cosmologies to sample.

In [ ]:
NN_noise_realizations = 2

Cosmological prior range for training

In [ ]:
dict_bounds_cosmo_train = dict(
    A = [-0.9, -0.1],
    B = [1.6, 2.],
)
NN_samples_cosmo_train = 2048
seed_train = 0

Cosmological prior range for test

In [ ]:
dict_bounds_cosmo_test = dict(
    A = [-0.9, -0.1],
    B = [1.6, 2.],
)
NN_samples_cosmo_test = 1024
seed_test = 2

##### Define dictionaries with baryonic parameters

In [ ]:
dict_bounds_augs_ModelA = dict(
    C = [-2., -1.7]
)

dict_bounds_augs_ModelB = dict(
    C = [-0.2, -0.1]
)

dict_bounds_augs_Modelall = dict(
    C = [-2., 2.]
)

# Generate training set

In [ ]:
ds_mode = "TRAIN"
path_save = os.path.join(path_save_root, ds_mode)

NN_samples_cosmo = NN_samples_cosmo_train
dict_bounds_cosmo = dict_bounds_cosmo_train
seed = seed_train

In [ ]:
dict_bounds_augs = dict_bounds_augs_ModelA
model_name = "ModelA"
NN_samples_augs = 3

cosmos, xx, aug_params = generate_toymodel_dataset(
    NN_samples_cosmo      = NN_samples_cosmo,
    NN_samples_augs       = NN_samples_augs,
    NN_noise_realizations = NN_noise_realizations,
    dict_bounds_cosmo     = dict_bounds_cosmo,
    dict_bounds_augs      = dict_bounds_augs,
    seed                  = seed,
    path_save             = path_save,
    model_name            = model_name,
    kcut                  = kcut,
    ks                    = ks,
    kmin                  = kmin,
    kmax                  = kmax,
    NN_k                  = NN_k
)

In [ ]:
dict_bounds_augs = dict_bounds_augs_ModelB
model_name = "ModelB"
NN_samples_augs = 3

cosmos, xx, aug_params = generate_toymodel_dataset(
    NN_samples_cosmo      = NN_samples_cosmo,
    NN_samples_augs       = NN_samples_augs,
    NN_noise_realizations = NN_noise_realizations,
    dict_bounds_cosmo     = dict_bounds_cosmo,
    dict_bounds_augs      = dict_bounds_augs,
    seed                  = seed,
    path_save             = path_save,
    model_name            = model_name,
    kcut                  = kcut,
    ks                    = ks,
    kmin                  = kmin,
    kmax                  = kmax,
    NN_k                  = NN_k
)

In [ ]:
dict_bounds_augs = dict_bounds_augs_Modelall
model_name = "Modelall"
NN_samples_augs = 3

cosmos, xx, aug_params = generate_toymodel_dataset(
    NN_samples_cosmo      = NN_samples_cosmo,
    NN_samples_augs       = NN_samples_augs,
    NN_noise_realizations = NN_noise_realizations,
    dict_bounds_cosmo     = dict_bounds_cosmo,
    dict_bounds_augs      = dict_bounds_augs,
    seed                  = seed,
    path_save             = path_save,
    model_name            = model_name,
    kcut                  = kcut,
    ks                    = ks,
    kmin                  = kmin,
    kmax                  = kmax,
    NN_k                  = NN_k
)

# Generate test set

In [ ]:
ds_mode = "TEST"
path_save = os.path.join(path_save_root, ds_mode)

NN_samples_cosmo = NN_samples_cosmo_test
dict_bounds_cosmo = dict_bounds_cosmo_test
seed = seed_test

In [ ]:
dict_bounds_augs = dict_bounds_augs_ModelA
model_name = "ModelA"
NN_samples_augs = 3

cosmos, xx, aug_params = generate_toymodel_dataset(
    NN_samples_cosmo      = NN_samples_cosmo,
    NN_samples_augs       = NN_samples_augs,
    NN_noise_realizations = NN_noise_realizations,
    dict_bounds_cosmo     = dict_bounds_cosmo,
    dict_bounds_augs      = dict_bounds_augs,
    seed                  = seed,
    path_save             = path_save,
    model_name            = model_name,
    kcut                  = kcut,
    ks                    = ks,
    kmin                  = kmin,
    kmax                  = kmax,
    NN_k                  = NN_k
)

In [ ]:
dict_bounds_augs = dict_bounds_augs_ModelB
model_name = "ModelB"
NN_samples_augs = 3

cosmos, xx, aug_params = generate_toymodel_dataset(
    NN_samples_cosmo      = NN_samples_cosmo,
    NN_samples_augs       = NN_samples_augs,
    NN_noise_realizations = NN_noise_realizations,
    dict_bounds_cosmo     = dict_bounds_cosmo,
    dict_bounds_augs      = dict_bounds_augs,
    seed                  = seed,
    path_save             = path_save,
    model_name            = model_name,
    kcut                  = kcut,
    ks                    = ks,
    kmin                  = kmin,
    kmax                  = kmax,
    NN_k                  = NN_k
)

In [ ]:
dict_bounds_augs = dict_bounds_augs_Modelall
model_name = "Modelall"
NN_samples_augs = 3

cosmos, xx, aug_params = generate_toymodel_dataset(
    NN_samples_cosmo      = NN_samples_cosmo,
    NN_samples_augs       = NN_samples_augs,
    NN_noise_realizations = NN_noise_realizations,
    dict_bounds_cosmo     = dict_bounds_cosmo,
    dict_bounds_augs      = dict_bounds_augs,
    seed                  = seed,
    path_save             = path_save,
    model_name            = model_name,
    kcut                  = kcut,
    ks                    = ks,
    kmin                  = kmin,
    kmax                  = kmax,
    NN_k                  = NN_k
)

In [ ]:
%matplotlib inline

In [ ]:
kk = np.linspace(kmin, kmax, NN_k)

In [ ]:
fig, ax = mpl.pyplot.subplots(1,1, figsize=(5,5))
ax.plot(kk, xx[:, 0, 0].T)

In [ ]:
fig, ax = mpl.pyplot.subplots(1,1, figsize=(5,5))
ax.plot(kk, xx[0, -1, :].T)

In [ ]:
# fig, ax = mpl.pyplot.subplots(1,1, figsize=(5,5))
# ax.plot(kk, xx[0, :, 0].T)